In [14]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
# import OS module
import os
# Get the list of all files and directories
path = "./exetat/"
dir_list = os.listdir(path)

out_binary_image = False
out_processed = False
out_cropped = False
f = open("./resultat.txt", "w")
# A function that returns the length of the value:
def order_dir_list(file_name):
    name = file_name.split('.')[0]
    return int(name)

dir_list.sort(key=order_dir_list)

for file_name in dir_list:
    image_path = path+file_name
    image_yolo = cv2.imread(image_path, cv2.IMREAD_COLOR)
    #image_resized = cv2.resize(image_yolo, (640, 640))

    #image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    model = YOLO("exetat_rdc.pt")
    # Redimensionner l'image pour YOLO
    #image_resized = cv2.resize(image, (640, 640))

    # Exécuter la détection d'objets avec YOLO
    results = model(image_yolo)

    x, y, w, h = 0, 0, 0, 0  
    x_x, y_y, w_w, h_h = 0, 0, 0, 0
    # Filtrer les résultats pour la classe ID = 1
    #matrice_code_candidat_bbox = None
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])  # Boîte englobante
            class_id = int(box.cls[0])  # Classe détectée
            if class_id == 3:  # On cherche la classe avec ID = 1
                x_x, y_y, w_w, h_h = (x1, y1, x2 - x1, y2 - y1)  # (x, y, w, h)
            elif class_id == 1:  # On cherche la classe avec ID = 1
                x, y, w, h = (x1, y1, x2 - x1, y2 - y1)  # Boîte englobante
                

    # Étape 1 : Appliquer un seuil pour binariser l'image
    # Binarisation avec méthode Otsu pour un seuil adaptatif
    _, thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # # Étape 2 : Détecter les contours dans l'image
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
    def processed_image_matricule(image, contour = 10):
        equalized_image = cv2.equalizeHist(image)
        # Étape 2 : Filtrage du bruit avec un filtre gaussien
        blurred_image = cv2.GaussianBlur(equalized_image, (1, 1), 0)

        # Étape 3 : Binarisation avec le seuil adaptatif
        binarized_image = cv2.adaptiveThreshold(
            blurred_image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 199, 5)

        # Étape 4 : Correction d'alignement (non nécessaire si l'image est bien alignée)
        # (Nous n'appliquons pas de transformation affine pour l'instant)

        # Étape 5 : Affiner les contours (optionnel)
        kernel = np.ones((contour, contour), np.uint8)
        processed_image = cv2.morphologyEx(binarized_image, cv2.MORPH_CLOSE, kernel)
        # cv2.imwrite("processed_image.png", processed_image)

        return processed_image

    def processed_image_reponse(image, contour = 10):
        equalized_image = cv2.equalizeHist(image)
        # Étape 2 : Filtrage du bruit avec un filtre gaussien
        blurred_image = cv2.blur(equalized_image, (2, 2), 0)

        # Étape 3 : Binarisation avec le seuil adaptatif
        binarized_image = cv2.adaptiveThreshold(
            blurred_image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 199, 5)

        # Étape 4 : Correction d'alignement (non nécessaire si l'image est bien alignée)
        # (Nous n'appliquons pas de transformation affine pour l'instant)

        # Étape 5 : Affiner les contours (optionnel)
        kernel = np.ones((contour, contour), np.uint8)
        processed_image = cv2.morphologyEx(binarized_image, cv2.MORPH_CLOSE, kernel)
        # cv2.imwrite("processed_image.png", processed_image)

        return processed_image

    # Fonction pour extraire une région spécifique d'une image
    def extract_region_matricule(image, x_y_w_h):
        x, y, w, h = x_y_w_h
        return image[y:y+h, x:x+w]
    
    def extract_region_reponse(image, x_y_w_h):
        x, y, w, h = x_y_w_h
        return image[y:y+h, x:x+w]

    def dechiffrer_reponse(matrice):
        matrice = np.array(matrice)
        code_dechiffre = [str(int(argmin)) for (argmin, min_c) in zip(np.argmin(matrice, axis=1) + 1, np.min(matrice, axis=1)) if min_c < 255]
        result = ''.join(code_dechiffre)
        return result

    def enregistrer_fichier_contour(image, zones_of_interest, nom_fichier_contour='temp'):
        # plt.figure(figsize=(10, 5))
        image_contours = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        for key, (x, y, w, h) in zones_of_interest.items():
            cv2.rectangle(image_contours, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Afficher l'image avec les contours détectés
        # plt.imshow(cv2.cvtColor(image_contours, cv2.COLOR_BGR2RGB))
        # # plt.title("Contours détectés")
        # plt.axis("off")
        # plt.show()
        # plt.savefig()
        cv2.imwrite(f'{nom_fichier_contour}.png', image_contours)

    def image_contour(image, zones_of_interest):
        # plt.figure(figsize=(10, 5))
        image_contours = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        for key, (x, y, w, h) in zones_of_interest.items():
            cv2.rectangle(image_contours, (x, y), (x + w, y + h), (0, 255, 0), 2)

        return image_contours

    def dechiffrer_code_candidat(matrice):
        matrice = np.array(matrice)
        rows, cols = matrice.shape
        code_dechiffre = [str(int(argmin)) for (argmin, min_c) in zip(np.argmin(matrice, axis=0), np.min(matrice, axis=0)) if min_c < 255]
        result = ''.join(code_dechiffre)
        return result

    def analyser_matrice_reponse(image_thresh,rows,cols, margin_seuil = 0.4, orientation_par_colonne = True):
        #
        height, width = image_thresh.shape
        cell_height = height // rows
        cell_width = width // cols

        # Marges pour ignorer les bordures
        margin_height = int(cell_height * margin_seuil)  # 40% de hauteur
        # margin_width = int(cell_width * margin_seuil)   # 40% de largeur
        margin_width = int(cell_width * margin_seuil * 0)   # 40% de largeur

        # Analyser chaque cellule
        matrice = []
        matrice_1 = []
        for i in range(rows):
            ligne = []
            ligne_1 = []
            for j in range(cols):
                # Extraire une cellule en ignorant les bordures
                cell = image_thresh[
                    i * cell_height + margin_height:(i + 1) * cell_height - margin_height,
                    j * cell_width + margin_width:(j + 1) * cell_width - margin_width
                ]
                matrice_1 = 1
                # Calculer l'intensité moyenne
                mean_intensity = np.mean(cell)
                # Déterminer si c'est une case noire (0) ou blanche (1)
                ligne.append(round(float(mean_intensity), 2))
            matrice.append(ligne)

        # f.write(f'\n{file_name}\n________________________________________________\n{matrice}'.replace('], ','], \n'))

        code_dechiffre =  dechiffrer_code_candidat(matrice = matrice) if orientation_par_colonne else dechiffrer_reponse(matrice = matrice)
        return code_dechiffre

    def analyser_matrice_matricule(image_thresh,rows,cols, margin_seuil = 0.4, orientation_par_colonne = True):
        #
        # Binarisation avec méthode Otsu pour un seuil adaptatif
        # Sauvegarder l'image binaire pour vérification

        # print(image_thresh.shape)
        height, width = image_thresh.shape
        cell_height = height // rows
        cell_width = width // cols

        # Marges pour ignorer les bordures
        margin_height = int(cell_height * margin_seuil)  # 40% de hauteur
        margin_width = int(cell_width * margin_seuil)   # 40% de largeur

        # Analyser chaque cellule
        matrice = []
        for i in range(rows):
            ligne = []
            for j in range(cols):
                # Extraire une cellule en ignorant les bordures
                cell = image_thresh[
                    i * cell_height + margin_height:(i + 1) * cell_height - margin_height,
                    j * cell_width + margin_width:(j + 1) * cell_width - margin_width
                ]

                # Calculer l'intensité moyenne
                mean_intensity = np.mean(cell)

                # Déterminer si c'est une case noire (0) ou blanche (1)
                seuil = 210 #if orientation_par_colonne else 168
                ligne.append(1 if mean_intensity > seuil else 0)
                # ligne.append(round(float(mean_intensity), 2))
            matrice.append(ligne)

        code_dechiffre = ''

        if orientation_par_colonne :
            for j in range(cols):  # Parcourir les colonnes
                # cases_noires = [i for i in range(rows) if matrice[i][j] == 0]  # Lignes avec cases noires
                cases_noires = [str(i) for i in range(rows) if matrice[i][j] == 0]
                code_dechiffre = code_dechiffre + ''.join(cases_noires)
                # for i in range(rows):
                #     if matrice[i][j] == 0:
                #         code_dechiffre.append(str(i))
                # print(f"Colonne {j}: Cases noires aux indices {cases_noires}")
        else:
            # print(matrice)
            for i in range(rows):  # Parcourir les ligne
                # cases_noires = [str(j+1) for j in range(cols) if matrice[i][j] == 0] # Lignes avec cases noires
                # code_dechiffre = code_dechiffre + ''.join(cases_noires)

                for j in range(cols):
                    if matrice[i][j] == 0:
                        code_dechiffre = code_dechiffre + str(j+1)
                        # code_dechiffre.append(str(j+1))
                # print(f"Colonne {i}: Cases noires aux indices {cases_noires}")

        #
        # code_dechiffre = ''
        # if orientation_par_colonne :
        #     for j in range(cols):  # Parcourir les colonnes
        #         cases_noires = [str(i) for i in range(rows) if matrice[i][j] == 0] # Lignes avec cases noires
        #         code_dechiffre = code_dechiffre + ''.join(cases_noires)
        # else:
        #     for i in range(rows):  # Parcourir les ligne
        #         cases_noires = [str(j+1) for j in range(cols) if matrice[i][j] == 0] # Lignes avec cases noires
        #         code_dechiffre = code_dechiffre + ''.join(cases_noires)
        return code_dechiffre


    def cropped_image(image_crop, rows = 10, cols = 14, margin_seuil = 0.25):
        # Seuillage pour isoler les cases noires (adaptation si nécessaire)
        _, thresh = cv2.threshold(image_crop, 100, 255, cv2.THRESH_BINARY_INV)
        # Détection des contours pour les cases noires
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Trouver les coordonnées minimales et maximales des contours détectés (rect englobant)
        x_min, y_min, x_max, y_max = np.inf, np.inf, -np.inf, -np.inf
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            # Exclure les petits bruits en vérifiant les dimensions des cases
            # if 30 <= w <= 40 and 41 <= h <= 50:  # Taille minimale des cases noires
            x_min = min(x_min, x)
            y_min = min(y_min, y)
            x_max = max(x_max, x + w)
            y_max = max(y_max, y + h)

        # Calcul des dimensions de la grille (10 lignes, 14 colonnes)
        # rows,cols, margin_seuil = 10, 14, 0.25

        grid_height = (y_max - y_min)*(1 + margin_seuil) / 10  # Hauteur d'une ligne
        grid_width = (x_max - x_min) / 14  # Largeur d'une colonne

        # Convertir en entiers pour les tracés
        grid_height = int(round(grid_height))
        grid_width = int(round(grid_width))
        grid_image = cv2.cvtColor(image_crop, cv2.COLOR_GRAY2BGR)

        cell_height = rows
        cell_width = cols
        # Tracer les lignes horizontales (10 lignes)
        for i in range(rows + 1):  # 10 lignes => 11 positions (y_min + i * grid_height)
            y = y_min + i * grid_height
            cv2.line(grid_image, (x_min, y), (x_max, y), (0, 255, 0), 1)
    
        # Tracer les lignes verticales (14 colonnes)
        for j in range(cols + 1):  # 14 colonnes => 15 positions (x_min + j * grid_width)
            x = x_min + j * grid_width
            cv2.line(grid_image, (x, y_min), (x, y_max), (0, 255, 0), 1)

        # calculer la hauteur maximale de découper

        y_max = y_min + rows * grid_height

        grid_cropped = grid_image[y_min:y_max, x_min:x_max]
        cv2.imwrite(f"./matrices/grid/{file}.png", grid_cropped)
        return image_crop[y_min:y_max, x_min:x_max]

    
    # Définir des zones clés pour l'extraction visuelle, basées sur l'observation
    zones_of_interest  = {
        #"matrice_code_candidat": (300, 2355, 1210, 670),  # Exemple
        "matrice_code_candidat": (x, y, w, h),  # Exemple
        "code_candidat": (300, 2200, 1210, 90),  # Exemple
        "matrice_reponses": (x_x, y_y, w_w, h_h),
        "code_epreuve": (2770, 330, 800, 90),
        "nom_eleve": (1780, 170, 2300, 90),
        # "code_barre": (50, 750, 400, 50)
    }
    

    resultat = ''
    margin_seuil = 0.4
    enregistrer_fichier_contour(image, zones_of_interest, nom_fichier_contour='temp')
    matrice_code_candidat_binary_image = extract_region_matricule(image = image, x_y_w_h = zones_of_interest["matrice_code_candidat"])
    matrice_matrice_reponse_binary_image = extract_region_reponse(image = image, x_y_w_h = zones_of_interest["matrice_reponses"])
    
    contour_image_binary = image_contour(image = image, zones_of_interest = zones_of_interest)
    
    cv2.imwrite("matrice_code_candidat_binary_image.png", matrice_code_candidat_binary_image)
    matrice_code_candidat_processed_image = processed_image_matricule(matrice_code_candidat_binary_image, contour = 10)
    cv2.imwrite("matrice_code_candidat_processed_image.png", matrice_code_candidat_processed_image)
    resultat_matrice_code = analyser_matrice_matricule(image_thresh = matrice_code_candidat_processed_image, rows = 10, cols = 14, margin_seuil = 0)

    #print(f'{file_name} => {resultat_matrice_code} ==> {len(resultat_matrice_code)}')

    
    resultat_matrice_reponse = analyser_matrice_reponse(image_thresh = matrice_matrice_reponse_binary_image, rows = 30, cols = 6, margin_seuil = margin_seuil, orientation_par_colonne = False)
    resultat_exetat = f'{file_name} --- {resultat_matrice_reponse} ==> {len(resultat_matrice_reponse)} ----- {resultat_matrice_code} ==> {len(resultat_matrice_code)}  '
    #resultat = f'{file_name},{resultat_matrice_reponse},{len(resultat_matrice_reponse)},{resultat_matrice_code},{len(resultat_matrice_code)}'
    resultat = f'{file_name} - {resultat_matrice_reponse} ==> {len(resultat_matrice_reponse)} , {resultat_matrice_code} ==> {len(resultat_matrice_code)}'

    print(resultat_exetat)
    #print(resultat)
    print(f"Coordonnées de la Zone Matrice_Réponse : {x_x, y_y, w_w, h_h}")
    print(f"Coordonnées de la Zone Matrice_Matricule : {x, y, w, h}")
    f.write(f'\n{resultat}')
f.close()


0: 480x640 1 code_barre, 1 matrice_matricule, 1 matricule, 1 reponse, 792.9ms
Speed: 14.6ms preprocess, 792.9ms inference, 3.5ms postprocess per image at shape (1, 3, 480, 640)
3.png --- 213515621635123533343416311455 ==> 30 ----- 14051201130046 ==> 14  
Coordonnées de la Zone Matrice_Réponse : (4273, 456, 540, 2939)
Coordonnées de la Zone Matrice_Matricule : (280, 2357, 1256, 676)

0: 480x640 1 code_barre, 1 matrice_matricule, 1 matricule, 1 reponse, 385.6ms
Speed: 25.5ms preprocess, 385.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)
4.png --- 213515621635123533343416341455 ==> 30 ----- 14051201130276 ==> 14  
Coordonnées de la Zone Matrice_Réponse : (4275, 467, 533, 2924)
Coordonnées de la Zone Matrice_Matricule : (284, 2360, 1240, 678)

0: 480x640 1 code_barre, 1 matrice_matricule, 1 matricule, 1 reponse, 446.8ms
Speed: 28.0ms preprocess, 446.8ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
5.png --- 243545421422223336341254362145 ==> 30 --